# Notebook for running basic sanity checks on the DFT results

In [29]:
import os
import re
import copy
import numpy as np
import glob
import sys

import rmgpy.kinetics

DFT_DIR = os.path.join(os.environ['AUTOSCIENCE_REPO'], 'dft')
sys.path.append(DFT_DIR)
import autotst_wrapper

sys.path.append(os.environ['DATABASE_DIR'])
import database_fun

## Load all of the calculations

In [30]:
# load the calculations
# gather all of the calculations
thermo_libs = glob.glob(os.path.join(DFT_DIR, 'thermo', 'species*', 'arkane', 'RMG_libraries'))

# Load the Arkane thermo
entries = []
for i, lib_path in enumerate(thermo_libs):
    matches = re.search('species_([0-9]{4})', lib_path)
    species_index = int(matches[1])
    ark_thermo_database = rmgpy.data.thermo.ThermoDatabase()
    ark_thermo_database.load_libraries(
        lib_path,
    )

    for key in ark_thermo_database.libraries['thermo'].entries.keys():
        entry = ark_thermo_database.libraries['thermo'].entries[key]
        entry.index = species_index
        entry.label = entry.item.smiles
        entries.append(entry)
print(f'{len(entries)} thermo entries')

sp_entries = [e.item for e in entries]


# first, get valid kinetics from old workflow
kinetics_libs = glob.glob(os.path.join(DFT_DIR, 'kinetics', 'reaction*', 'arkane', 'RMG_libraries'))

# Load the Arkane kinetics
k_entries = []
for i, lib_path in enumerate(kinetics_libs):
    
    matches = re.search('reaction_([0-9]{4,6})', lib_path)
    reaction_index = int(matches[1])
    
    ark_kinetics_database = rmgpy.data.kinetics.KineticsDatabase()
    ark_kinetics_database.load_libraries(lib_path)
    
    
    
    
    # TODO fix bug related to load_libraries not getting the actual name
    for key in ark_kinetics_database.libraries[''].entries.keys():
        entry = ark_kinetics_database.libraries[''].entries[key]
        
        
        # check isomorphism with include_list
        idx = database_fun.get_unique_reaction_index(ark_kinetics_database.libraries[''].entries[key].item)

        entry.index = reaction_index
        k_entries.append(entry)
#         print(f'Adding\t{entry.index}\t{entry}')
print(f'{len(k_entries)} kinetics entries')

rxn_entries = [e.item for e in k_entries]

108 thermo entries
47 kinetics entries


## 1. Compare results to other methods

### 1.1 Do single point energies with AE corrections differ by more than 3.35 kcal/mol between m06-2x and dlpno-CCSD(T)?

In [31]:
np.sqrt(np.float_power(3.0, 2) + np.float_power(1.5, 2))  # uncertainties of m06-2x and dlpno-CCSD(T) added in quadrature

3.3541019662496847

### 1.2. Do any of the species thermodynamics differ by more than the GAV + dlpno-CCSD(T) uncertainty?

### 1.3 Do any of the reaction kinetics differ by more than the rate rule + dlpno-CCSD(T) uncertainty?

## 2. Check parameter values in reasonable range

In [44]:
def get_i_thing(thing, thing_list):
    for i in range(len(thing_list)):
        if thing.is_isomorphic(thing_list[i]):
            return i
    return -1

def get_reverse_reaction(reaction):
    assert reaction.kinetics is not None
    rev_reaction = copy.deepcopy(reaction)
    tmp_reactants = rev_reaction.reactants
    rev_reaction.reactants = rev_reaction.products
    rev_reaction.products = tmp_reactants
    rev_reaction.kinetics = reaction.generate_reverse_rate_coefficient()
    return rev_reaction

In [45]:
my_reactions = []
for i, k in enumerate(k_entries):
    
    r_fwd = rmgpy.reaction.Reaction()
    r_fwd.reactants = k_entries[i].item.reactants
    r_fwd.kinetics = k_entries[i].data
    for j in range(len(r_fwd.reactants)):
        index1 = get_i_thing(r_fwd.reactants[j].molecule[0], sp_entries)
        assert index1 >= 0
        
        r_fwd.reactants[j].thermo = entries[index1].data
    
    
    r_fwd.products = k_entries[i].item.products
    for j in range(len(r_fwd.products)):
        index1 = get_i_thing(r_fwd.products[j].molecule[0], sp_entries)
        assert index1 >= 0
        
        r_fwd.products[j].thermo = entries[index1].data
    
    r_rev = get_reverse_reaction(r_fwd)
    
    
    my_reactions.append([r_fwd, r_rev])

### 2.1 Negative reaction barriers?

In [46]:
negative_reactions = []
for r_fwd, r_rev in my_reactions:
    if r_fwd.kinetics.Ea.value_si <= 0 or r_rev.kinetics.Ea.value_si <= 0:
        negative_reactions.append(database_fun.get_unique_reaction_index(r_fwd))

if negative_reactions:
    print(f'You have {len(negative_reactions)} negative reaction barriers:')
    for i in negative_reactions:
        print(f'\t{i}')
else:
    print('All reaction barriers >= 0')

You have 15 negative reaction barriers:
	288
	404
	804
	808
	1288
	4721
	4728
	4729
	4778
	4779
	4796
	5046
	9358
	10105
	10106


### 2.2 $|n| \geq 4.0$ ?

In [47]:
large_n = []
N_THRESHOLD = 4.0
for r_fwd, r_rev in my_reactions:
    if np.abs(r_fwd.kinetics.n.value_si) >= N_THRESHOLD or np.abs(r_rev.kinetics.n.value_si) >= N_THRESHOLD:
        large_n.append(database_fun.get_unique_reaction_index(r_fwd))

if large_n:
    print(f'You have {len(large_n)} reactions with n >= {N_THRESHOLD}:')
    for i in large_n:
        print(f'\t{i}')
else:
    print(f'All reaction kinetics have |n| <= {N_THRESHOLD}')

You have 22 reactions with n >= 4.0:
	253
	280
	286
	321
	419
	714
	808
	1111
	1287
	1288
	4728
	4729
	4736
	4778
	4779
	4796
	5046
	9060
	10105
	10106
	10111
	10168


### 2.3 Any of the parameters have ridiculous exponents?

In [58]:
UPPER_LIMIT_A = 1e18
large_exp_A = []
for r_fwd, r_rev in my_reactions:
    if np.abs(r_fwd.kinetics.A.value_si) >= UPPER_LIMIT_A or np.abs(r_rev.kinetics.A.value_si) >= UPPER_LIMIT_A:
        large_exp_A.append(database_fun.get_unique_reaction_index(r_fwd))

if large_exp_A:
    print(f'You have {len(large_exp_A)} reactions with A >= {UPPER_LIMIT_A}:')
    for i in large_exp_A:
        print(f'\t{i}')
else:
    print(f'All reaction kinetics have |A| <= {UPPER_LIMIT_A}')
    
UPPER_LIMIT_EA = 3e6  # J/mol
large_exp = []
for r_fwd, r_rev in my_reactions:
    if np.abs(r_fwd.kinetics.Ea.value_si) >= UPPER_LIMIT_EA or np.abs(r_rev.kinetics.Ea.value_si) >= UPPER_LIMIT_EA:
        large_exp.append(database_fun.get_unique_reaction_index(r_fwd))

if large_exp:
    print(f'You have {len(large_exp)} reactions with A >= {UPPER_LIMIT_EA}:')
    for i in large_exp:
        print(f'\t{i}')
else:
    print(f'All reaction kinetics have |Ea| <= {UPPER_LIMIT_EA}')

All reaction kinetics have |A| <= 1e+18
All reaction kinetics have |Ea| <= 3000000.0


### 2.4 Collision rate violators?

## 3. Errors from Software Components

### 3.1 Any final ts/conformer Gaussian files with 14+ atoms missing the iop(2/9=2000) keyword?

### 3.2 Did Arkane complain about any of the rotor energies being less than the lowest energy conformer?

## 4. Other

### 4.1 Flying Hydrogen atoms? (incorrect atom numbering on hindered rotor scans)